# Depth Anything 3 (DA3) Usage Example

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bogarrichard/Depth-Anything-3/blob/dev/notebooks/da3.ipynb)

This notebook demonstrates how to use Depth Anything 3 for camera poses and depth estimation.

In [ ]:
# Setup: works both in Colab (clones this fork fresh) and locally (just
# makes sure the working directory is the repo root, so the relative
# asset paths below resolve).
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.isdir("Depth-Anything-3"):
        !git clone --depth 1 --branch dev https://github.com/bogarrichard/Depth-Anything-3.git
    os.chdir("Depth-Anything-3")
    !pip install -q -e .
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import torch
from depth_anything_3.api import DepthAnything3
from depth_anything_3.utils.visualize import visualize_depth
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DepthAnything3.from_pretrained("depth-anything/DA3NESTED-GIANT-LARGE-1.1")
model = model.to(device)
model.eval()
print(f"Model loaded on {device}")

In [ ]:
# Load sample images and run inference
image_paths = [
    "assets/examples/SOH/000.png",
    "assets/examples/SOH/010.png"
]

prediction = model.inference(
    image=image_paths,
    process_res=504,
    process_res_method="upper_bound_resize",
)
print(f"Depth shape: {prediction.depth.shape}")
print(f"Extrinsics: {prediction.extrinsics.shape if prediction.extrinsics is not None else 'None'}")
print(f"Intrinsics: {prediction.intrinsics.shape if prediction.intrinsics is not None else 'None'}")

In [ ]:
# Visualize input images and depth maps
n_images = prediction.depth.shape[0]

fig, axes = plt.subplots(2, n_images, figsize=(12, 6))

if n_images == 1:
    axes = axes.reshape(2, 1)

for i in range(n_images):
    # Show original image
    if prediction.processed_images is not None:
        axes[0, i].imshow(prediction.processed_images[i])
    axes[0, i].set_title(f"Input {i+1}")
    axes[0, i].axis('off')

    # Show depth map
    depth_vis = visualize_depth(prediction.depth[i], cmap="Spectral")
    axes[1, i].imshow(depth_vis)
    axes[1, i].set_title(f"Depth {i+1}")
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()